# Indicadores SNIS

#### Este código vai extrair os indicadores e gerar gráfico de perdas. Não mexa nos nomes nem nas pastas dentro do zip, apenas extraia-os na pasta desejada.

##### Altere o município e o caminho abaixo conforme desejado:

In [8]:
import pandas as pd
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import unicodedata



In [14]:

# ============================================================
# CONFIGURAÇÕES
# ============================================================

MUNICIPIO_SELECIONADO = "Concordia"

# Use None para não filtrar prestador.
# Exemplo: PRESTADOR_SELECIONADO = "CASAN"
PRESTADOR_SELECIONADO = None

DIRETORIO = r"C:\Users\gabriel.coimbra\Desktop\SNIS_SINISA"

PASTA_RESULTADO = os.path.join(DIRETORIO, "Resultado")
os.makedirs(PASTA_RESULTADO, exist_ok=True)


INDICADORES_ESGOTO = [
    "município",
    "prestador",
    "uf",
    "es005",
    "es006",
    "es007",
    "es001",
    "es026",
    "in015",
    "in016",
    "es002",
    "es003",
    "es004",
]


# ============================================================
# EQUIVALÊNCIA SINISA -> SNIS
# ============================================================

MAPA_SINISA_PARA_SNIS = {
    # População
    "gte0023": "es001",   # População total atendida com esgotamento sanitário
    "gte0001": "es026",   # População urbana atendida com rede de esgotamento sanitário

    # Ligações / economias / rede
    "gte0003": "es002",   # Ligações ativas de esgoto
    "gte1001": "es004",   # Extensão da rede pública de esgotamento sanitário

    # Volumes
    "gte1002": "es005",   # Volume total de esgoto coletado
    "gte1014": "es006",   # Volume total de esgoto tratado
    "gte1006": "es007",   # Volume total de esgoto faturado

    # Indicadores
    "ies2002": "in015",   # Índice de coleta de esgoto
    "ies2004": "in016",   # Índice de tratamento de esgoto
}


COLUNAS_AUXILIARES_SINISA = [
    "gte0001",
    "gte0002",
    "gte0006",
    "gte0016",
    "gte0023",
]


### ÁGUA
#### Tabela com indicadores  "ge12a", "ag001", "ag021", "ag003", "ag005", "in009", "in022", "in049".

In [15]:
# ============================================================
# FUNÇÕES AUXILIARES
# ============================================================

def normalizar_texto(txt):
    if pd.isna(txt):
        return ""

    txt = str(txt).strip().lower()
    txt = unicodedata.normalize("NFKD", txt).encode("ascii", "ignore").decode("ascii")
    txt = re.sub(r"\s+", " ", txt)

    return txt


def normalizar_coluna(col):
    col = normalizar_texto(col)

    substituicoes = {
        "municipio": "município",
        "codigo do municipio": "codigo_municipio",
        "codigo municipio": "codigo_municipio",
        "codigo do ibge": "cod_ibge",
        "cod_ibge": "cod_ibge",
        "uf": "uf",
        "nome do prestador de servicos": "prestador",
        "nome do prestador": "prestador",
        "sigla": "sigla",
        "macrorregiao": "macrorregião",
        "rm / ride": "rm_ride",
        "rm/ride": "rm_ride",
    }

    return substituicoes.get(col, col)


def tornar_colunas_unicas(cols):
    saida = []
    vistos = {}

    for col in cols:
        col = normalizar_coluna(col)

        if col in ["", "nan"]:
            col = "coluna_sem_nome"

        n = vistos.get(col, 0)

        if n == 0:
            saida.append(col)
        else:
            saida.append(f"{col}_{n}")

        vistos[col] = n + 1

    return saida


def extrair_ano(nome_arquivo):
    match = re.search(r"(20\d{2})", nome_arquivo)
    return match.group(1) if match else "Desconhecido"


def identificar_base(nome_arquivo):
    nome = nome_arquivo.lower()

    if "sinisa" in nome:
        return "SINISA"

    if "snis" in nome:
        return "SNIS"

    ano = extrair_ano(nome)

    if ano in ["2023", "2024"]:
        return "SINISA"

    return "SNIS"


def identificar_tipo(nome_arquivo):
    nome = nome_arquivo.lower()

    if "info" in nome:
        return "Info"

    if "ind" in nome:
        return "Ind"

    return "Ind"


def converter_numero(valor):
    if pd.isna(valor):
        return np.nan

    if isinstance(valor, (int, float, np.number)):
        return float(valor)

    valor = str(valor).strip()

    if valor == "":
        return np.nan

    valor_norm = normalizar_texto(valor)

    termos_invalidos = [
        "nao calculado",
        "campo vazio",
        "nao se aplica",
        "consultar",
    ]

    if any(t in valor_norm for t in termos_invalidos):
        return np.nan

    valor = valor.replace("%", "").replace(" ", "")

    if "," in valor:
        valor = valor.replace(".", "").replace(",", ".")

    try:
        return float(valor)
    except:
        return np.nan


def filtrar_municipio_prestador(df, municipio, prestador=None):
    if "município" not in df.columns:
        return pd.DataFrame()

    df = df.copy()

    df["município"] = df["município"].astype(str).str.strip()

    municipio_norm = normalizar_texto(municipio)

    df = df[df["município"].apply(normalizar_texto) == municipio_norm].copy()

    if prestador is not None and "prestador" in df.columns:
        prestador_norm = normalizar_texto(prestador)

        df = df[
            df["prestador"]
            .apply(normalizar_texto)
            .str.contains(prestador_norm, na=False, regex=False)
        ].copy()

    return df


def juntar_por_chaves(lista_df):
    lista_df = [
        df.copy()
        for df in lista_df
        if df is not None and not df.empty
    ]

    if not lista_df:
        return pd.DataFrame()

    resultado = lista_df[0]

    chaves_possiveis = ["município", "ano", "uf", "prestador", "sigla"]

    for df in lista_df[1:]:
        chaves_merge = [
            c for c in chaves_possiveis
            if c in resultado.columns and c in df.columns
        ]

        if not chaves_merge:
            chaves_merge = [
                c for c in ["município", "ano"]
                if c in resultado.columns and c in df.columns
            ]

        resultado = pd.merge(
            resultado,
            df,
            on=chaves_merge,
            how="outer",
            suffixes=("", "_dup")
        )

        cols_dup = [c for c in resultado.columns if c.endswith("_dup")]

        for col_dup in cols_dup:
            col_base = col_dup.replace("_dup", "")

            if col_base in resultado.columns:
                resultado[col_base] = resultado[col_base].combine_first(resultado[col_dup])

        resultado = resultado.drop(columns=cols_dup, errors="ignore")

    return resultado


# ============================================================
# LEITOR SNIS
# ============================================================

def ler_snis(file_path, municipio, indicadores_saida, prestador=None):
    nome_arquivo = os.path.basename(file_path)
    ano = extrair_ano(nome_arquivo)
    tipo = identificar_tipo(nome_arquivo)

    xls = pd.ExcelFile(file_path)
    dfs_abas = []

    for sheet_name in xls.sheet_names:
        if "observa" in normalizar_texto(sheet_name):
            continue

        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        if len(df_raw) < 12:
            continue

        if tipo == "Info":
            header_row_1 = df_raw.iloc[7]
            header_row_2 = df_raw.iloc[11]
            start_row = 12
        else:
            header_row_1 = df_raw.iloc[7]
            header_row_2 = df_raw.iloc[9]
            start_row = 10

        headers = list(header_row_1[:10]) + list(header_row_2[10:])
        headers = tornar_colunas_unicas(headers)

        df = df_raw.iloc[start_row:].copy()
        df.columns = headers[:len(df.columns)]
        df = df.dropna(how="all")

        if "município" not in df.columns:
            continue

        df["ano"] = ano
        df["base"] = "SNIS"
        df["tipo_arquivo"] = tipo
        df["arquivo_origem"] = nome_arquivo
        df["aba_origem"] = sheet_name

        df = filtrar_municipio_prestador(df, municipio, prestador)

        if df.empty:
            continue

        colunas_base = [
            "município",
            "ano",
            "uf",
            "prestador",
            "sigla",
            "base",
            "tipo_arquivo",
            "arquivo_origem",
            "aba_origem",
        ]

        colunas_desejadas = list(dict.fromkeys(colunas_base + indicadores_saida))
        colunas_existentes = [c for c in colunas_desejadas if c in df.columns]

        dfs_abas.append(df[colunas_existentes].copy())

    return juntar_por_chaves(dfs_abas)


# ============================================================
# LEITOR SINISA
# ============================================================

def detectar_linha_codigo_sinisa(df_raw):
    for i in range(len(df_raw)):
        valores = [normalizar_coluna(v) for v in df_raw.iloc[i].tolist()]

        tem_municipio = "município" in valores

        tem_codigo = any(
            re.match(r"^(cad|dfe|gte|ies|ife)\d+", str(v))
            for v in valores
        )

        if tem_municipio and tem_codigo:
            return i

    return None


def ler_sinisa(file_path, municipio, indicadores_saida, prestador=None):
    nome_arquivo = os.path.basename(file_path)
    ano = extrair_ano(nome_arquivo)
    tipo = identificar_tipo(nome_arquivo)

    xls = pd.ExcelFile(file_path)
    dfs_abas = []

    codigos_sinisa_necessarios = []

    for cod_sinisa, cod_snis in MAPA_SINISA_PARA_SNIS.items():
        if cod_snis in indicadores_saida:
            codigos_sinisa_necessarios.append(cod_sinisa)

    codigos_sinisa_necessarios += COLUNAS_AUXILIARES_SINISA

    for sheet_name in xls.sheet_names:
        if "nota" in normalizar_texto(sheet_name):
            continue

        df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        linha_codigo = detectar_linha_codigo_sinisa(df_raw)

        if linha_codigo is None:
            continue

        headers = df_raw.iloc[linha_codigo].tolist()
        headers = tornar_colunas_unicas(headers)

        df = df_raw.iloc[linha_codigo + 1:].copy()
        df.columns = headers[:len(df.columns)]
        df = df.dropna(how="all")

        if "município" not in df.columns:
            continue

        if "cad0005" in df.columns and "prestador" not in df.columns:
            df["prestador"] = df["cad0005"]

        if "cad0006" in df.columns and "sigla" not in df.columns:
            df["sigla"] = df["cad0006"]

        df["ano"] = ano
        df["base"] = "SINISA"
        df["tipo_arquivo"] = tipo
        df["arquivo_origem"] = nome_arquivo
        df["aba_origem"] = sheet_name

        df = filtrar_municipio_prestador(df, municipio, prestador)

        if df.empty:
            continue

        colunas_base = [
            "município",
            "ano",
            "uf",
            "prestador",
            "sigla",
            "base",
            "tipo_arquivo",
            "arquivo_origem",
            "aba_origem",
        ]

        colunas_para_manter = list(dict.fromkeys(
            colunas_base + codigos_sinisa_necessarios
        ))

        colunas_existentes = [
            c for c in colunas_para_manter
            if c in df.columns
        ]

        dfs_abas.append(df[colunas_existentes].copy())

    df_sinisa = juntar_por_chaves(dfs_abas)

    if df_sinisa.empty:
        return pd.DataFrame()

    for cod_sinisa, cod_snis in MAPA_SINISA_PARA_SNIS.items():
        if cod_sinisa in df_sinisa.columns:
            if cod_snis in df_sinisa.columns:
                df_sinisa[cod_snis] = df_sinisa[cod_snis].combine_first(df_sinisa[cod_sinisa])
            else:
                df_sinisa[cod_snis] = df_sinisa[cod_sinisa]

    # ES001:
    # Em 2024 existe GTE0023.
    # Em 2023 pode ser calculado como GTE0001 + GTE0002.
    if "es001" in indicadores_saida:
        calculo_es001 = None

        if "gte0001" in df_sinisa.columns and "gte0002" in df_sinisa.columns:
            calculo_es001 = (
                df_sinisa["gte0001"].apply(converter_numero).fillna(0)
                + df_sinisa["gte0002"].apply(converter_numero).fillna(0)
            )

            calculo_es001 = calculo_es001.replace(0, np.nan)

        if "es001" not in df_sinisa.columns:
            df_sinisa["es001"] = calculo_es001
        elif calculo_es001 is not None:
            df_sinisa["es001"] = (
                df_sinisa["es001"]
                .apply(converter_numero)
                .combine_first(calculo_es001)
            )

    # ES003:
    # No SINISA, economias urbanas e rurais aparecem separadas.
    if "es003" in indicadores_saida:
        if "gte0006" in df_sinisa.columns and "gte0016" in df_sinisa.columns:
            calculo_es003 = (
                df_sinisa["gte0006"].apply(converter_numero).fillna(0)
                + df_sinisa["gte0016"].apply(converter_numero).fillna(0)
            )

            calculo_es003 = calculo_es003.replace(0, np.nan)

            if "es003" not in df_sinisa.columns:
                df_sinisa["es003"] = calculo_es003
            else:
                df_sinisa["es003"] = (
                    df_sinisa["es003"]
                    .apply(converter_numero)
                    .combine_first(calculo_es003)
                )

    colunas_base_final = [
        "município",
        "ano",
        "uf",
        "prestador",
        "sigla",
        "base",
        "tipo_arquivo",
        "arquivo_origem",
        "aba_origem",
    ]

    colunas_finais = list(dict.fromkeys(colunas_base_final + indicadores_saida))
    colunas_existentes = [
        c for c in colunas_finais
        if c in df_sinisa.columns
    ]

    return df_sinisa[colunas_existentes].copy()


# ============================================================
# PROCESSAMENTO GERAL
# ============================================================

def processar_base_snis_sinisa(
    diretorio,
    municipio,
    indicadores_saida,
    prestador=None,
    exportar=True,
    nome_saida="Dados-Esgoto-Tabela.xlsx"
):
    arquivos_excel = [
        os.path.join(diretorio, f)
        for f in os.listdir(diretorio)
        if f.lower().endswith(".xlsx")
        and not f.startswith("~$")
        and ("SNIS" in f.upper() or "SINISA" in f.upper())
    ]

    dfs_info = []
    dfs_ind = []

    for arquivo in sorted(arquivos_excel):
        nome = os.path.basename(arquivo)
        base = identificar_base(nome)
        tipo = identificar_tipo(nome)

        print(f"Processando: {nome} | Base: {base} | Tipo: {tipo}")

        if base == "SNIS":
            df = ler_snis(
                file_path=arquivo,
                municipio=municipio,
                indicadores_saida=indicadores_saida,
                prestador=prestador
            )
        else:
            df = ler_sinisa(
                file_path=arquivo,
                municipio=municipio,
                indicadores_saida=indicadores_saida,
                prestador=prestador
            )

        if df.empty:
            print(f"  ⚠ Nenhum dado encontrado em {nome}")
            continue

        print(f"  ✅ {len(df)} linha(s) encontrada(s)")

        if tipo == "Info":
            dfs_info.append(df)
        else:
            dfs_ind.append(df)

    df_info = juntar_por_chaves(dfs_info)
    df_ind = juntar_por_chaves(dfs_ind)

    chaves_merge = ["município", "ano"]

    if "prestador" in df_info.columns and "prestador" in df_ind.columns:
        chaves_merge.append("prestador")

    if not df_info.empty and not df_ind.empty:
        df_final = pd.merge(
            df_info,
            df_ind,
            on=chaves_merge,
            how="outer",
            suffixes=("", "_dup")
        )

        cols_dup = [c for c in df_final.columns if c.endswith("_dup")]

        for col_dup in cols_dup:
            col_base = col_dup.replace("_dup", "")

            if col_base in df_final.columns:
                df_final[col_base] = df_final[col_base].combine_first(df_final[col_dup])

        df_final = df_final.drop(columns=cols_dup, errors="ignore")

    elif not df_info.empty:
        df_final = df_info.copy()

    elif not df_ind.empty:
        df_final = df_ind.copy()

    else:
        df_final = pd.DataFrame()

    if df_final.empty:
        print(f"⚠ Nenhum dado encontrado para {municipio}.")
        return df_final

    indicadores_validos = [
        c for c in indicadores_saida
        if c in df_final.columns and c not in ["município", "prestador", "uf"]
    ]

    if indicadores_validos:
        df_final = df_final.dropna(how="all", subset=indicadores_validos)

    if "ano" in df_final.columns:
        df_final["ano"] = df_final["ano"].astype(str)
        df_final = df_final.sort_values("ano")

    colunas_inicio = [
        "município",
        "ano",
        "uf",
        "prestador",
        "sigla",
        "base",
    ]

    colunas_ordenadas = []

    for c in colunas_inicio + indicadores_saida:
        if c in df_final.columns and c not in colunas_ordenadas:
            colunas_ordenadas.append(c)

    outras_colunas = [
        c for c in df_final.columns
        if c not in colunas_ordenadas
    ]

    df_final = df_final[colunas_ordenadas + outras_colunas]

    if exportar:
        output_path = os.path.join(PASTA_RESULTADO, f"{municipio}_{nome_saida}")
        df_final.to_excel(output_path, index=False)
        print(f"📂 Arquivo exportado para: {output_path}")

    return df_final


# ============================================================
# GRÁFICO DE ESGOTO
# ============================================================

def gerar_grafico_esgoto(df_final, municipio, diretorio_resultado):
    df_plot = df_final.copy()

    colunas_necessarias = ["ano", "es005", "es006", "es007"]

    faltantes = [
        c for c in colunas_necessarias
        if c not in df_plot.columns
    ]

    if faltantes:
        print(f"⚠ Não foi possível gerar o gráfico. Colunas ausentes: {faltantes}")
        return

    df_plot["ano"] = df_plot["ano"].astype(int)

    for col in ["es005", "es006", "es007"]:
        df_plot[col] = df_plot[col].apply(converter_numero)

    df_plot = df_plot.dropna(subset=["ano"])
    df_plot = df_plot.sort_values("ano")

    if df_plot.empty:
        print("⚠ Não há dados válidos para o gráfico.")
        return

    fig, ax1 = plt.subplots(figsize=(10, 5))

    bar_width = 0.2
    x_vals = df_plot["ano"].astype(int)

    bars1 = ax1.bar(
        x_vals - 0.2,
        df_plot["es005"],
        width=bar_width,
        label="ES005 - Volume de esgotos coletado"
    )

    bars2 = ax1.bar(
        x_vals,
        df_plot["es006"],
        width=bar_width,
        label="ES006 - Volume de esgotos tratado"
    )

    bars3 = ax1.bar(
        x_vals + 0.2,
        df_plot["es007"],
        width=bar_width,
        label="ES007 - Volume de esgotos faturado"
    )

    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()

            if pd.notna(height):
                ax1.text(
                    bar.get_x() + bar.get_width() / 2,
                    height,
                    f"{height:,.0f}".replace(",", "."),
                    ha="center",
                    va="bottom",
                    fontsize=10,
                    fontweight="bold"
                )

    ax1.set_ylabel("1.000 m³/ano")
    ax1.set_xlabel("Ano")
    ax1.set_xticks(x_vals)
    ax1.set_xticklabels(x_vals, fontsize=10)
    ax1.tick_params(axis="x", length=0)
    ax1.tick_params(axis="y", length=0)
    ax1.grid(axis="y", alpha=0.2)

    legend = fig.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, 0),
        ncol=3,
        fontsize=10
    )

    legend.get_frame().set_linewidth(0)

    output_png = os.path.join(
        diretorio_resultado,
        f"{municipio}_Grafico-Esgoto.png"
    )

    plt.savefig(output_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"📊 Gráfico exportado para: {output_png}")


# ============================================================
# EXECUÇÃO
# ============================================================

df_final = processar_base_snis_sinisa(
    diretorio=DIRETORIO,
    municipio=MUNICIPIO_SELECIONADO,
    indicadores_saida=INDICADORES_ESGOTO,
    prestador=PRESTADOR_SELECIONADO,
    exportar=True,
    nome_saida="Dados-Esgoto-Tabela.xlsx"
)

display(df_final)

gerar_grafico_esgoto(
    df_final=df_final,
    municipio=MUNICIPIO_SELECIONADO,
    diretorio_resultado=PASTA_RESULTADO
)

⚠ Nenhum dado encontrado para Concordia.


""


⚠ Não foi possível gerar o gráfico. Colunas ausentes: ['ano', 'es005', 'es006', 'es007']
